# Agent-1: Theoretical Foundations of Choi Representations

This notebook develops the four standard finite-dimensional representations of a quantum channel: Kraus, Choi, Stinespring, and natural/Liouville form. We use the project convention

$$C_{\mathcal{E}} = \sum_{i,j} |i\rangle\langle j| \otimes \mathcal{E}(|i\rangle\langle j|),$$

so the input system is the first tensor factor and the output system is the second tensor factor. References: Nielsen and Chuang, Chapter 8; Watrous, Chapter 2; Wilde, Chapter 4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from channel_reps import (
    amplitude_damping_channel,
    apply_channel,
    bit_flip_channel,
    choi_rank,
    choi_to_kraus,
    choi_to_natural,
    compose_channels_choi,
    depolarizing_channel,
    identity_channel,
    is_cp,
    is_tp,
    is_unital,
    kraus_to_choi,
    kraus_to_stinespring,
    natural_to_choi,
    pauli_channel,
    phase_damping_channel,
    phase_flip_channel,
    random_channel,
    stinespring_to_kraus,
)

np.set_printoptions(precision=3, suppress=True)
np.random.seed(42)

PALETTE = {
    "blue": "#1f77b4",
    "orange": "#ff7f0e",
    "green": "#2ca02c",
    "red": "#d62728",
    "purple": "#9467bd",
    "gray": "#7f7f7f",
}


## 1. Channel Representations

A channel can be represented as Kraus operators, a Choi matrix, a Stinespring isometry, or a natural superoperator. The examples below are one-qubit channels, so each Choi matrix is `4 x 4`.

In [ ]:
def summarize_channel(name, kraus_ops):
    choi = kraus_to_choi(kraus_ops)
    eigvals = np.linalg.eigvalsh(choi)
    print(f"\n{name}")
    print("Choi matrix:")
    print(choi)
    print(f"CP={is_cp(choi)}, TP={is_tp(choi, d_in=2)}, rank={choi_rank(choi)}")
    print(f"eigenvalues={np.round(eigvals, 6)}")
    return choi

examples = {
    "identity": identity_channel(2),
    "bit flip p=0.2": bit_flip_channel(0.2),
    "phase flip p=0.2": phase_flip_channel(0.2),
    "depolarizing p=0.3": depolarizing_channel(0.3),
    "amplitude damping gamma=0.3": amplitude_damping_channel(0.3),
    "phase damping gamma=0.3": phase_damping_channel(0.3),
}

choi_examples = {name: summarize_channel(name, kraus) for name, kraus in examples.items()}


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 6), constrained_layout=True)
for ax, (name, choi) in zip(axes.ravel(), choi_examples.items()):
    im = ax.imshow(np.real(choi), cmap="RdBu_r", vmin=-2, vmax=2)
    ax.set_title(name, fontsize=9)
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.75, label="Re(C)")
plt.show()


## 2. Choi-Jamiolkowski Isomorphism

The isomorphism sends a linear map to the result of applying it to half of the unnormalized maximally entangled operator. In coordinates, each block of `C` is the channel output on a matrix unit. Conversely, diagonalizing a positive Choi matrix recovers Kraus operators: if `C = sum_r lambda_r |v_r><v_r|`, reshape each `sqrt(lambda_r) v_r` into a Kraus operator.

In [ ]:
random_kraus = random_channel(d_in=2, d_out=2, n_kraus=3)
random_choi = kraus_to_choi(random_kraus)
recovered_kraus = choi_to_kraus(random_choi)

psi = np.array([1, 1j], dtype=complex)
psi = psi / np.linalg.norm(psi)
rho = np.outer(psi, psi.conj())
out_original = apply_channel(rho, random_kraus)
out_recovered = apply_channel(rho, recovered_kraus)

print("Recovered Kraus count:", len(recovered_kraus))
print("Choi rank:", choi_rank(random_choi))
print("||E(rho) - E_recovered(rho)||_F =", np.linalg.norm(out_original - out_recovered))
print("CP:", is_cp(random_choi), "TP:", is_tp(random_choi, d_in=2))


## 3. Properties Revealed by the Choi Matrix

For the input-first convention, complete positivity is `C >= 0`, trace preservation is `Tr_output(C) = I_input`, and the numerical rank of `C` is the minimum Kraus count. Unitality for square channels is `Tr_input(C) = I_output`.

In [ ]:
for name in ["depolarizing p=0.3", "amplitude damping gamma=0.3"]:
    choi = choi_examples[name]
    print(f"{name}: min eigenvalue={np.min(np.linalg.eigvalsh(choi)):.6g}, ", end="")
    print(f"TP={is_tp(choi, 2)}, unital={is_unital(choi, 2)}, Choi rank={choi_rank(choi)}")

bad_choi = choi_examples["identity"].copy()
bad_choi[0, 0] = -0.1
print("Perturbed identity Choi is CP:", is_cp(bad_choi))


Entanglement-breaking channels are exactly those whose normalized Choi states are separable across input-output. A full separability test is outside this folder's dependency budget, but the completely depolarizing qubit channel has a separable Choi state `I/2 \otimes I`, making it an instructive special case.

In [ ]:
completely_depolarizing = depolarizing_channel(1.0)
C_dep = kraus_to_choi(completely_depolarizing)
print("Completely depolarizing Choi / d_in:")
print(C_dep / 2)
print("This normalized Choi state has maximally mixed output blocks and is separable for the qubit depolarizing endpoint.")


## 4. Conversions Between Representations

The module implements direct conversions `Kraus <-> Choi`, `Kraus <-> Stinespring`, and `Choi <-> Natural`. The remaining pairings are obtained by composition through these direct conversions. The natural representation uses column-stacking vectorization, `vec(E(rho)) = S vec(rho)`.

In [ ]:
kraus = random_channel(d_in=2, d_out=3, n_kraus=2)
choi = kraus_to_choi(kraus)
natural = choi_to_natural(choi)
choi_again = natural_to_choi(natural)
stinespring = kraus_to_stinespring(kraus)
kraus_again = stinespring_to_kraus(stinespring, env_dim=2)

print("Non-square Choi shape:", choi.shape)
print("Natural shape:", natural.shape)
print("Stinespring isometry shape:", stinespring.shape)
print("Choi <-> Natural error:", np.linalg.norm(choi - choi_again))
print("Kraus <-> Stinespring block error:", sum(np.linalg.norm(a - b) for a, b in zip(kraus, kraus_again)))


## 5. Composition and Channel Algebra

In natural form, composition is ordinary matrix multiplication: `S_{E2 o E1} = S_E2 S_E1`. The Choi link product is the same operation after reshuffling between Choi and natural indices; `compose_channels_choi` exposes this safely.

In [ ]:
p = 0.2
q = 0.3
C1 = kraus_to_choi(bit_flip_channel(p))
C2 = kraus_to_choi(bit_flip_channel(q))
C_composed = compose_channels_choi(C1, C2)

expected_p = p + q - 2 * p * q
C_expected = kraus_to_choi(bit_flip_channel(expected_p))

print("Expected composed bit-flip probability:", expected_p)
print("Composition error:", np.linalg.norm(C_composed - C_expected))
print("CP:", is_cp(C_composed), "TP:", is_tp(C_composed, d_in=2))
